In [30]:
import os
import time
from getpass import getpass

import pandas as pd
import requests

MASTER_FILE = "master_wq_station_table.csv"
METADATA_FILE = "usgs_28stations_metadata.csv"

OUTPUT_DIR = "usgs_continuous_download"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Working directory:")
print(os.getcwd())

Working directory:
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph


In [31]:
master_station_table = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

if "wq_station" in master_station_table.columns:
    id_col = "wq_station"

elif "monitoring_location_id" in master_station_table.columns:
    id_col = "monitoring_location_id"

else:
    raise ValueError(
        "Could not find station ID column."
    )

selected_wq_stations = (
    master_station_table
    .copy()
    .set_index(id_col)
)

selected_wq_stations.index = (
    selected_wq_stations.index
    .astype(str)
    .str.strip()
)

print(
    "Selected stations:",
    len(selected_wq_stations)
)

print(
    selected_wq_stations[
        "region"
    ].value_counts()
)

Selected stations: 28
region
Top half       22
Bottom half     6
Name: count, dtype: int64


In [32]:
BOOL_COLUMNS = [
    "DO concentration",
    "DO saturation",
    "Water temperature",
    "Specific conductance",
    "Turbidity",
    "pH",
    "Nitrate",
    "Chlorophyll-a",
    "fDOM/CDOM",
    "Discharge",
    "Stage",
    "has_hydrology"
]


def clean_bool(value):

    if pd.isna(value):
        return False

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in [
        "true",
        "1",
        "yes"
    ]


for col in BOOL_COLUMNS:

    if col in selected_wq_stations.columns:

        selected_wq_stations[col] = (
            selected_wq_stations[col]
            .apply(clean_bool)
        )

In [33]:
flowing_metadata = pd.read_csv(
    METADATA_FILE,
    low_memory=False
)

print(
    "Metadata rows:",
    len(flowing_metadata)
)

print(
    "Metadata stations:",
    flowing_metadata[
        "monitoring_location_id"
    ].astype(str).nunique()
)

print(
    flowing_metadata.columns.tolist()
)

Metadata rows: 204
Metadata stations: 28
['id', 'unit_of_measure', 'parameter_name', 'parameter_code', 'statistic_id', 'last_modified', 'begin', 'end', 'computation_period_identifier', 'computation_identifier', 'thresholds', 'sublocation_identifier', 'primary', 'monitoring_location_id', 'web_description', 'parameter_description', 'parent_time_series_id', 'data_gap_interval', 'statistics_begin', 'variable']


In [34]:
PCODE_TO_VARIABLE = {
    "00300": "DO concentration",
    "00301": "DO saturation",
    "00010": "Water temperature",
    "00095": "Specific conductance",
    "63680": "Turbidity",
    "00400": "pH",
    "99133": "Nitrate",
    "32316": "Chlorophyll-a",
    "32315": "Chlorophyll-a",
    "32295": "fDOM/CDOM",
    "32322": "fDOM/CDOM",
    "00060": "Discharge",
    "00065": "Stage"
}


flowing_metadata[
    "parameter_code"
] = (
    flowing_metadata[
        "parameter_code"
    ]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False
    )
    .str.zfill(5)
)


flowing_metadata[
    "variable"
] = (
    flowing_metadata[
        "parameter_code"
    ]
    .map(
        PCODE_TO_VARIABLE
    )
)

In [35]:
possible_begin_cols = [
    "begin_utc",
    "begin",
    "start",
    "start_time"
]

possible_end_cols = [
    "end_utc",
    "end",
    "end_time"
]


begin_col = next(
    (
        col
        for col in possible_begin_cols
        if col in flowing_metadata.columns
    ),
    None
)

end_col = next(
    (
        col
        for col in possible_end_cols
        if col in flowing_metadata.columns
    ),
    None
)


print(
    "Begin column:",
    begin_col
)

print(
    "End column:",
    end_col
)


if begin_col is None or end_col is None:

    raise ValueError(
        "Could not find metadata "
        "begin/end columns."
    )


flowing_metadata[
    "begin_dt"
] = pd.to_datetime(
    flowing_metadata[begin_col],
    errors="coerce",
    utc=True
)


flowing_metadata[
    "end_dt"
] = pd.to_datetime(
    flowing_metadata[end_col],
    errors="coerce",
    utc=True
)

Begin column: begin
End column: end


In [36]:
VARIABLE_PCODES = {
    "DO concentration": "00300",
    "Water temperature": "00010",
    "Specific conductance": "00095",
    "Turbidity": "63680",
    "pH": "00400",
    "Discharge": "00060",
    "Stage": "00065"
}


def station_download_window(
    site_id,
    variables
):

    meta = flowing_metadata[
        (
            flowing_metadata[
                "monitoring_location_id"
            ]
            .astype(str)
            .str.strip()
            == str(site_id)
        )
        &
        (
            flowing_metadata[
                "variable"
            ].isin(variables)
        )
    ].copy()

    starts = []
    ends = []

    for variable in variables:

        rows = meta[
            meta["variable"]
            == variable
        ]

        if rows.empty:
            continue

        start = rows[
            "begin_dt"
        ].min()

        end = rows[
            "end_dt"
        ].max()

        if pd.notna(start):
            starts.append(start)

        if pd.notna(end):
            ends.append(end)

    if not starts or not ends:
        return None, None

    common_start = max(
        starts
    )

    common_end = min(
        ends
    )

    if (
        pd.isna(common_start)
        or pd.isna(common_end)
        or common_start >= common_end
    ):
        return None, None

    download_start = max(
        common_start,
        common_end
        - pd.Timedelta(
            days=365
        )
    )

    return (
        download_start,
        common_end
    )

In [37]:
download_plan = {}


for site_id in (
    selected_wq_stations
    .index
    .astype(str)
):

    info = (
        selected_wq_stations
        .loc[site_id]
    )

    variables = []

    for variable in [
        "DO concentration",
        "Water temperature",
        "Specific conductance",
        "Turbidity",
        "pH"
    ]:

        if info.get(
            variable,
            False
        ):
            variables.append(
                variable
            )

    if info.get(
        "Discharge",
        False
    ):

        variables.append(
            "Discharge"
        )

    elif info.get(
        "Stage",
        False
    ):

        variables.append(
            "Stage"
        )

    start, end = (
        station_download_window(
            site_id,
            variables
        )
    )

    if start is None:

        print(
            "No common window:",
            site_id
        )

        continue

    download_plan[
        site_id
    ] = {
        "variables":
            variables,

        "start":
            start,

        "end":
            end
    }


print(
    "Selected stations:",
    len(selected_wq_stations)
)

print(
    "Stations in plan:",
    len(download_plan)
)

Selected stations: 28
Stations in plan: 28


In [38]:
plan_rows = []

for site_id, info in (
    download_plan.items()
):

    for variable in (
        info["variables"]
    ):

        plan_rows.append({
            "station_id":
                site_id,

            "variable":
                variable,

            "parameter_code":
                VARIABLE_PCODES[
                    variable
                ],

            "start":
                info["start"],

            "end":
                info["end"]
        })


download_plan_df = pd.DataFrame(
    plan_rows
)


download_plan_df.to_csv(
    "usgs_download_plan.csv",
    index=False
)


print(
    "Saved plan rows:",
    len(download_plan_df)
)

Saved plan rows: 139


In [39]:
USGS_API_KEY = "Wg8C6eEgjnC9TdHHYwnA1lBZXdJNVpYk5SDcH5Dr"

In [40]:
if (
    "USGS_API_KEY"
    not in globals()
    or not USGS_API_KEY
):

    USGS_API_KEY = getpass(
        "Enter USGS API key: "
    ).strip()


CONTINUOUS_URL = (
    "https://api.waterdata.usgs.gov/"
    "ogcapi/v1/collections/"
    "continuous/items"
)


session = requests.Session()

session.headers.update({
    "X-Api-Key":
        USGS_API_KEY
})


PAGE_LIMIT = 50000

In [41]:
def download_series(
    site_id,
    parameter_code,
    start,
    end
):

    params = {
        "f":
            "json",

        "limit":
            PAGE_LIMIT,

        "monitoring_location_id":
            str(site_id),

        "parameter_code":
            str(
                parameter_code
            ).zfill(5),

        "time":
            (
                f"{pd.Timestamp(start).isoformat()}/"
                f"{pd.Timestamp(end).isoformat()}"
            )
    }

    rows = []

    next_url = (
        CONTINUOUS_URL
    )

    first_page = True
    page = 0

    while next_url:

        page += 1

        while True:

            if first_page:

                response = session.get(
                    next_url,
                    params=params,
                    timeout=180
                )

            else:

                response = session.get(
                    next_url,
                    timeout=180
                )

            if (
                response.status_code
                == 429
            ):

                wait = int(
                    response.headers.get(
                        "Retry-After",
                        60
                    )
                )

                print(
                    f"Rate limited. "
                    f"Waiting {wait}s..."
                )

                time.sleep(
                    wait
                )

                continue

            if (
                response.status_code
                in [
                    500,
                    502,
                    503,
                    504
                ]
            ):

                print(
                    "USGS server error. "
                    "Waiting 30 seconds..."
                )

                time.sleep(
                    30
                )

                continue

            response.raise_for_status()

            break

        payload = (
            response.json()
        )

        features = payload.get(
            "features",
            []
        )

        print(
            f"    Page {page}: "
            f"{len(features):,} rows"
        )

        for feature in features:

            rows.append(
                feature.get(
                    "properties",
                    {}
                )
            )

        next_url = None

        for link in payload.get(
            "links",
            []
        ):

            if (
                link.get("rel")
                == "next"
            ):

                next_url = (
                    link.get(
                        "href"
                    )
                )

                break

        first_page = False

        time.sleep(
            0.1
        )

    return pd.DataFrame(
        rows
    )

In [42]:
download_plan_df = pd.read_csv(
    "usgs_download_plan.csv"
)

download_plan_df[
    "start"
] = pd.to_datetime(
    download_plan_df[
        "start"
    ],
    utc=True
)

download_plan_df[
    "end"
] = pd.to_datetime(
    download_plan_df[
        "end"
    ],
    utc=True
)


for i, row in (
    download_plan_df
    .iterrows()
):

    site_id = str(
        row["station_id"]
    )

    variable = (
        row["variable"]
    )

    parameter_code = str(
        row[
            "parameter_code"
        ]
    ).zfill(5)

    start = row[
        "start"
    ]

    end = row[
        "end"
    ]

    station_dir = (
        os.path.join(
            OUTPUT_DIR,
            site_id
        )
    )

    os.makedirs(
        station_dir,
        exist_ok=True
    )

    safe_variable = (
        variable
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )

    final_file = (
        os.path.join(
            station_dir,
            f"{safe_variable}.csv"
        )
    )

    if (
        os.path.exists(
            final_file
        )
        and
        os.path.getsize(
            final_file
        ) > 0
    ):

        print(
            f"[{i+1}/"
            f"{len(download_plan_df)}] "
            f"SKIP {site_id} "
            f"{variable}"
        )

        continue

    print()
    print(
        f"[{i+1}/"
        f"{len(download_plan_df)}]"
    )

    print(
        site_id,
        variable
    )

    print(
        start.date(),
        "to",
        end.date()
    )

    try:

        df = download_series(
            site_id=
                site_id,

            parameter_code=
                parameter_code,

            start=
                start,

            end=
                end
        )

        temp_file = (
            final_file
            + ".tmp"
        )

        df.to_csv(
            temp_file,
            index=False
        )

        os.replace(
            temp_file,
            final_file
        )

        print(
            f"Saved "
            f"{len(df):,} rows"
        )

    except Exception as e:

        print(
            "FAILED:",
            site_id,
            variable
        )

        print(e)

        continue


[1/139]
USGS-09070500 DO concentration
2025-10-02 to 2026-10-02
    Page 1: 35,004 rows
Saved 35,004 rows

[2/139]
USGS-09070500 Water temperature
2025-10-02 to 2026-10-02
    Page 1: 35,009 rows
Saved 35,009 rows

[3/139]
USGS-09070500 Specific conductance
2025-10-02 to 2026-10-02
    Page 1: 34,856 rows
Saved 34,856 rows

[4/139]
USGS-09070500 Turbidity
2025-10-02 to 2026-10-02
    Page 1: 31,099 rows
Saved 31,099 rows

[5/139]
USGS-09070500 pH
2025-10-02 to 2026-10-02
    Page 1: 35,005 rows
Saved 35,005 rows

[6/139]
USGS-09070500 Discharge
2025-10-02 to 2026-10-02
    Page 1: 33,359 rows
Saved 33,359 rows

[7/139]
USGS-09071750 DO concentration
2025-10-02 to 2026-10-02
    Page 1: 34,926 rows
Saved 34,926 rows

[8/139]
USGS-09071750 Water temperature
2025-10-02 to 2026-10-02
    Page 1: 34,982 rows
Saved 34,982 rows

[9/139]
USGS-09071750 Specific conductance
2025-10-02 to 2026-10-02
    Page 1: 34,960 rows
Saved 34,960 rows

[10/139]
USGS-09071750 Turbidity
2025-10-02 to 2026-10

In [44]:
results = []

for _, row in (
    download_plan_df
    .iterrows()
):

    site_id = str(
        row["station_id"]
    )

    variable = (
        row["variable"]
    )

    safe_variable = (
        variable
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )

    file_path = os.path.join(
        OUTPUT_DIR,
        site_id,
        f"{safe_variable}.csv"
    )

    exists = (
        os.path.exists(
            file_path
        )
        and
        os.path.getsize(
            file_path
        ) > 0
    )

    results.append({
        "station":
            site_id,

        "variable":
            variable,

        "downloaded":
            exists
    })


progress = pd.DataFrame(
    results
)


print(
    progress[
        "downloaded"
    ].value_counts()
)


print(
    "\nCompleted:",
    progress[
        "downloaded"
    ].sum(),
    "/",
    len(progress)
)

downloaded
True    139
Name: count, dtype: int64

Completed: 139 / 139


In [45]:
import os
import pandas as pd

audit_rows = []

for _, row in download_plan_df.iterrows():

    site_id = str(row["station_id"])
    variable = row["variable"]

    safe_variable = (
        variable
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )

    file_path = os.path.join(
        OUTPUT_DIR,
        site_id,
        f"{safe_variable}.csv"
    )

    try:
        df = pd.read_csv(
            file_path,
            low_memory=False
        )

        if "time" in df.columns:
            times = pd.to_datetime(
                df["time"],
                errors="coerce",
                utc=True
            )

            start_time = times.min()
            end_time = times.max()

        else:
            start_time = pd.NaT
            end_time = pd.NaT

        audit_rows.append({
            "station_id": site_id,
            "variable": variable,
            "rows": len(df),
            "start": start_time,
            "end": end_time
        })

    except Exception as e:

        audit_rows.append({
            "station_id": site_id,
            "variable": variable,
            "rows": -1,
            "start": pd.NaT,
            "end": pd.NaT,
            "error": str(e)
        })


download_audit = pd.DataFrame(
    audit_rows
)

download_audit

,station_id,variable,rows,start,end,error
0,USGS-09070500,DO concentration,35004,2025-10-02 04:00:00+00:00,2026-10-02 04:00:00+00:00,NaN
1,USGS-09070500,Water temperature,35009,2025-10-02 04:00:00+00:00,2026-10-02 04:00:00+00:00,NaN
2,USGS-09070500,Specific conductance,34856,2025-10-02 04:00:00+00:00,2026-10-02 04:00:00+00:00,NaN
3,USGS-09070500,Turbidity,31099,2025-10-02 04:00:00+00:00,2026-10-02 04:00:00+00:00,NaN
4,USGS-09070500,pH,35005,2025-10-02 04:00:00+00:00,2026-10-02 04:00:00+00:00,NaN
...,...,...,...,...,...,...
134,USGS-393259107194801,Water temperature,104363,2025-10-02 10:00:00+00:00,2026-10-02 10:00:00+00:00,NaN
135,USGS-393259107194801,Specific conductance,104292,2025-10-02 10:00:00+00:00,2026-10-02 10:00:00+00:00,NaN
136,USGS-393259107194801,Turbidity,102282,2025-10-02 10:00:00+00:00,2026-10-02 10:00:00+00:00,NaN
137,USGS-393259107194801,pH,100933,2025-10-02 10:00:00+00:00,2026-10-02 10:00:00+00:00,NaN


In [46]:
print("Series downloaded:", len(download_audit))

print(
    "Series with data:",
    (download_audit["rows"] > 0).sum()
)

print(
    "Zero-row series:",
    (download_audit["rows"] == 0).sum()
)

print(
    "Failed reads:",
    (download_audit["rows"] < 0).sum()
)

print("\nRow count summary:")
print(
    download_audit["rows"]
    .describe()
)

Series downloaded: 139
Series with data: 136
Zero-row series: 0
Failed reads: 3

Row count summary:
count       139.000000
mean      36848.625899
std       25126.656651
min          -1.000000
25%       30823.000000
50%       34400.000000
75%       35003.500000
max      104997.000000
Name: rows, dtype: float64


In [47]:
download_audit[
    [
        "station_id",
        "variable",
        "rows",
        "start",
        "end"
    ]
].sort_values(
    "rows"
).head(30)

,station_id,variable,rows,start,end
98,USGS-09365000,Discharge,-1,NaT,NaT
121,USGS-09404200,Water temperature,-1,NaT,NaT
122,USGS-09404200,Specific conductance,-1,NaT,NaT
101,USGS-09368000,Turbidity,4045,2023-09-11 00:00:00+00:00,2023-11-14 16:00:00+00:00
106,USGS-09371010,Turbidity,4100,2023-09-11 00:00:00+00:00,2023-11-15 15:00:00+00:00
99,USGS-09368000,Water temperature,4190,2023-09-11 00:00:00+00:00,2023-11-14 16:00:00+00:00
100,USGS-09368000,Specific conductance,4192,2023-09-11 00:00:00+00:00,2023-11-14 16:00:00+00:00
102,USGS-09368000,pH,4192,2023-09-11 00:00:00+00:00,2023-11-14 16:00:00+00:00
107,USGS-09371010,pH,4285,2023-09-11 00:00:00+00:00,2023-11-15 15:00:00+00:00
105,USGS-09371010,Specific conductance,4285,2023-09-11 00:00:00+00:00,2023-11-15 15:00:00+00:00


In [48]:
download_audit.to_csv(
    "usgs_download_audit.csv",
    index=False
)

print("Saved usgs_download_audit.csv")

Saved usgs_download_audit.csv


In [49]:
failed = download_audit[
    download_audit["rows"] < 0
].copy()

print(
    failed[
        [
            "station_id",
            "variable",
            "error"
        ]
    ].to_string(index=False)
)

   station_id             variable                         error
USGS-09365000            Discharge No columns to parse from file
USGS-09404200    Water temperature No columns to parse from file
USGS-09404200 Specific conductance No columns to parse from file


In [50]:
failed_pairs = [
    ("USGS-09365000", "Discharge"),
    ("USGS-09404200", "Water temperature"),
    ("USGS-09404200", "Specific conductance")
]

for site_id, variable in failed_pairs:

    row = download_plan_df[
        (
            download_plan_df["station_id"].astype(str)
            == site_id
        )
        &
        (
            download_plan_df["variable"]
            == variable
        )
    ].iloc[0]

    pcode = str(
        row["parameter_code"]
    ).zfill(5)

    start = pd.to_datetime(
        row["start"],
        utc=True
    )

    end = pd.to_datetime(
        row["end"],
        utc=True
    )

    print("\n" + "=" * 70)
    print(site_id, variable)
    print("Parameter:", pcode)
    print("Window:", start, "to", end)

    df = download_series(
        site_id=site_id,
        parameter_code=pcode,
        start=start,
        end=end
    )

    print(
        "Rows returned:",
        len(df)
    )

    if len(df) == 0:
        print(
            "NO DATA returned for this "
            "station/variable/window."
        )
        continue

    station_dir = os.path.join(
        OUTPUT_DIR,
        site_id
    )

    safe_variable = (
        variable
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )

    final_file = os.path.join(
        station_dir,
        f"{safe_variable}.csv"
    )

    temp_file = (
        final_file + ".tmp"
    )

    df.to_csv(
        temp_file,
        index=False
    )

    os.replace(
        temp_file,
        final_file
    )

    print(
        "Saved:",
        final_file
    )


USGS-09365000 Discharge
Parameter: 00060
Window: 2020-10-05 19:00:00+00:00 to 2021-10-05 19:00:00+00:00
    Page 1: 0 rows
Rows returned: 0
NO DATA returned for this station/variable/window.

USGS-09404200 Water temperature
Parameter: 00010
Window: 2025-10-02 09:30:00+00:00 to 2026-10-02 09:30:00+00:00
    Page 1: 0 rows
Rows returned: 0
NO DATA returned for this station/variable/window.

USGS-09404200 Specific conductance
Parameter: 00095
Window: 2025-10-02 09:30:00+00:00 to 2026-10-02 09:30:00+00:00
    Page 1: 0 rows
Rows returned: 0
NO DATA returned for this station/variable/window.
